# Lyft Motion Prediction — ResNet18 Inference

**Before running**
1. Add Input → competition `lyft-motion-prediction-autonomous-vehicles`
2. Add Input → Dataset with `resnet18_motion.pt` (+ optional `agent_motion_config.yaml`)
3. GPU On / Internet **Off**（若镜像无 `l5kit`，需在 Dataset 中 vendoring 或仅用允许的包）
4. Save & Run All → Submit

Note: Kaggle 竞赛镜像通常可 `pip install l5kit`（有网调试时）；正式提交 Internet Off，请把依赖放进 Dataset。

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import yaml
from torch import nn
from torch.utils.data import DataLoader
from torchvision.models import resnet18
from tqdm.auto import tqdm

from l5kit.data import ChunkedDataset, LocalDataManager
from l5kit.dataset import AgentDataset
from l5kit.evaluation import write_pred_csv
from l5kit.geometry import transform_points
from l5kit.rasterization import build_rasterizer

INPUT = Path("/kaggle/input")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def find_data_root() -> Path:
    for root in [
        INPUT / "lyft-motion-prediction-autonomous-vehicles",
        INPUT / "competitions" / "lyft-motion-prediction-autonomous-vehicles",
    ]:
        if (root / "scenes").is_dir() or (root / "meta.json").exists():
            return root
    for meta in INPUT.rglob("meta.json"):
        return meta.parent
    raise FileNotFoundError("Add competition data.")


DATA = find_data_root()
os.environ["L5KIT_DATA_FOLDER"] = str(DATA)
WEIGHTS = next(iter(INPUT.rglob("resnet18_motion.pt")))
cfg_files = list(INPUT.rglob("agent_motion_config.yaml"))
print("data", DATA)
print("weights", WEIGHTS)
print("device", DEVICE)

In [ ]:
DEFAULT_CFG = {
    "format_version": 4,
    "model_params": {
        "model_architecture": "resnet18",
        "history_num_frames": 10,
        "future_num_frames": 50,
        "step_time": 0.1,
        "render_ego_history": True,
    },
    "raster_params": {
        "raster_size": [224, 224],
        "pixel_size": [0.5, 0.5],
        "ego_center": [0.25, 0.5],
        "map_type": "py_semantic",
        "satellite_map_key": "aerial_map/aerial_map.png",
        "semantic_map_key": "semantic_map/semantic_map.pb",
        "dataset_meta_key": "meta.json",
        "filter_agents_threshold": 0.5,
        "disable_traffic_light_faces": False,
        "set_origin_to_bottom": True,
    },
    "test_data_loader": {
        "key": "scenes/test.zarr",
        "batch_size": 32,
        "shuffle": False,
        "num_workers": 2,
    },
}

if cfg_files:
    cfg = yaml.safe_load(cfg_files[0].read_text())
else:
    cfg = DEFAULT_CFG


def build_model(cfg):
    model = resnet18(weights=None)
    history = cfg["model_params"]["history_num_frames"]
    num_in = 3 + (history + 1) * 2
    model.conv1 = nn.Conv2d(num_in, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Linear(model.fc.in_features, 2 * cfg["model_params"]["future_num_frames"])
    return model


payload = torch.load(WEIGHTS, map_location="cpu")
model = build_model(cfg).to(DEVICE)
state = payload["model"] if isinstance(payload, dict) and "model" in payload else payload
model.load_state_dict(state)
model.eval()
print("loaded")

In [ ]:
dm = LocalDataManager(None)
rasterizer = build_rasterizer(cfg, dm)
test_key = cfg.get("test_data_loader", DEFAULT_CFG["test_data_loader"])["key"]
zarr = ChunkedDataset(dm.require(test_key)).open()

mask = None
for p in [DATA / "scenes" / "mask.npz", DATA / "mask.npz"]:
    if p.exists():
        mask = np.load(p)["arr_0"]
        print("mask", p, "sum", int(mask.sum()))
        break

dataset = AgentDataset(cfg, zarr, rasterizer, agents_mask=mask)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)
print("agents", len(dataset))

coords_list, timestamps, track_ids = [], [], []
with torch.no_grad():
    for batch in tqdm(loader):
        out = model(batch["image"].to(DEVICE)).reshape(batch["target_positions"].shape)
        agents_coords = out.cpu().numpy()
        world_from_agents = batch["world_from_agent"].numpy()
        centroids = batch["centroid"].numpy()
        coords_offset = transform_points(agents_coords, world_from_agents) - centroids[:, None, :2]
        coords_list.append(coords_offset)
        timestamps.append(batch["timestamp"].numpy().copy())
        track_ids.append(batch["track_id"].numpy().copy())

write_pred_csv(
    "submission.csv",
    timestamps=np.concatenate(timestamps),
    track_ids=np.concatenate(track_ids),
    coords=np.concatenate(coords_list),
)
print("wrote submission.csv")